# Exploración AML 2: Sensores de Caudal y Recurrencia Temporal

En el análisis topológico anterior identificamos las anomalías estáticas de la red y el vaciado absoluto (100%). Sin embargo, un motor de fraude robusto debe detectar estafadores adaptativos. 

En este notebook construiremos dos sensores dinámicos:
1. **Presión de salida (Ratio de vaciado):** Un medidor continuo del caudal relativo extraído, para detectar cuando los atacantes dejan fondos parciales (ej. 10%) buscando evadir reglas estrictas.
2. **Grosor de tubería (Recurrencia):** Medición de la frecuencia de transacciones entre un mismo par de nodos, diseñado para alertar sobre flujos de *layering* (lavado por goteo o capas).

## 1. Conexión a la Tubería Principal
Absorbemos el flujo del dataset ya procesado que contiene la estructura topológica (`in_degree`, `out_degree`) y el sensor binario original.

In [12]:
import pandas as pd
import numpy as np

print("Conectando a la salida del Notebook 1...")
# Leemos el parquet que ya trae el in_degree, out_degree y flag_vaciado
df = pd.read_parquet('../data/processed/paysim_features.parquet')

print(f"Flujo recibido con topología previa: {len(df)} registros")

Conectando a la salida del Notebook 1...
Flujo recibido con topología previa: 2770409 registros


## 2. Sensor de Vaciado Parcial (Caudal Relativo)

El fraude no siempre vacía la cuenta a cero. Para que el modelo XGBoost encuentre el umbral exacto de presión anómala, necesitamos una variable continua. 
Calcularemos la fracción extraída (`amount / oldbalanceOrg`). Esto nos revelará la distribución de presión en los ataques y si existen deciles de fraude operando bajo el radar de la regla del 100%.

In [13]:
# Si el saldo original es mayor a 0, calculamos la fracción extraída. Si es 0, dejamos 0.
df['ratio_vaciado'] = np.where(
    df['oldbalanceOrg'] > 0,
    df['amount'] / df['oldbalanceOrg'],
    0
)

# Revisamos cómo se distribuye este ratio en los fraudes
df_fraud = df[df['isFraud'] == 1]
print("Distribución del vaciado parcial en fraudes:")
display(df_fraud['ratio_vaciado'].describe(percentiles=[0.1, 0.2, 0.5, 0.8, 0.9, 0.99]).round(4))

Distribución del vaciado parcial en fraudes:


count    8213.0000
mean        0.9938
std         0.3908
min         0.0000
10%         1.0000
20%         1.0000
50%         1.0000
80%         1.0000
90%         1.0000
99%         1.0000
max        29.5266
Name: ratio_vaciado, dtype: float64

## 3. Recurrencia Temporal (Grosor de la Tubería)

Para identificar mulas de retención (nodos que transfieren sistemáticamente a una misma bóveda), agrupamos las transacciones por el par origen-destino. 
*Nota sobre los datos sintéticos (PaySim):* Este dataset simula ataques de "golpe y huida" (frecuencia = 1). No obstante, el sensor queda calibrado e integrado en la tubería para detectar estructuras reales de *layering* (goteo continuo) en entornos de producción.

In [14]:
# Contamos cuántas veces interactúan el mismo origen y destino
tuberias = df.groupby(['nameOrig', 'nameDest']).size().reset_index(name='frecuencia_transacciones')

# Acoplamos el sensor al dataset principal
df = df.merge(tuberias, on=['nameOrig', 'nameDest'], how='left')

print("Mulas con mayor recurrencia (alta frecuencia de transferencias):")
display(df.sort_values('frecuencia_transacciones', ascending=False)[['nameOrig', 'nameDest', 'frecuencia_transacciones', 'isFraud']].head())

Mulas con mayor recurrencia (alta frecuencia de transferencias):


,nameOrig,nameDest,frecuencia_transacciones,isFraud
0,C1305486145,C553264065,1,1
1,C840083671,C38997010,1,1
2,C905080434,C476402209,1,0
3,C1670993182,C1100439041,1,0
4,C1984094095,C932583850,1,0


## 4. Integración y Presurización a V2

Acoplamos los nuevos manómetros (caudal y recurrencia) al dataset principal. Exportamos el flujo resultante a formato Parquet, el cual servirá como la cámara de combustión final para entrenar nuestro motor XGBoost en el próximo paso.

In [15]:
# Celda 4: Exportación V2
ruta_exportacion = '../data/processed/paysim_features_v2.parquet'
df.to_parquet(ruta_exportacion, index=False)
print(f"Dataset V2 (Topología + Caudal + Recurrencia) presurizado y guardado en: {ruta_exportacion}")

Dataset V2 (Topología + Caudal + Recurrencia) presurizado y guardado en: ../data/processed/paysim_features_v2.parquet


## 5. Conclusiones del Ensamblaje Dinámico

Con la finalización de este notebook, el motor de detección cuenta con dos sensores dinámicos operativos:

1. **Válvula de Caudal (Ratio de Vaciado):** Al transformar la regla rígida del 100% en una variable continua, permitiremos que el algoritmo identifique fugas de presión parciales y detecte a los estafadores que intentan evadir umbrales fijos.
2. **Grosor de Tubería (Recurrencia):** Aunque la simulación actual refleja ataques de impacto único (frecuencia 1), la arquitectura lógica está lista para identificar redes de *layering* (goteo continuo hacia mulas de retención) al conectarse a flujos transaccionales reales.

**Próximo paso:** El dataset `paysim_features_v2.parquet` entrará a la cámara de combustión (Notebook 3), donde ensamblaremos la bomba de clasificación (XGBoost) y los manómetros de explicabilidad (SHAP).